
## Contig Generation from De Bruijn Graphs

**Topic:** Graph Algorithms and Genome Assembly

### Overview

This notebook explores the generation of contigs from collections of DNA k-mers using De Bruijn graphs.

Contigs are assembled sequences representing unambiguous paths through a graph. In genome assembly, maximal non-branching paths can be used to reconstruct contiguous sequences from overlapping fragments.

### Objectives

- Construct De Bruijn graphs from DNA k-mers.
- Calculate the in-degree and out-degree of graph nodes.
- Identify nodes with exactly one incoming and one outgoing edge.
- Generate contigs from non-branching paths.
- Investigate branching structures and isolated cycles.
- Validate contig generation using representative test cases.

### Implementation

The implementation follows the maximal non-branching paths approach to contig generation.

It includes:

- Construction of a De Bruijn graph from overlapping k-mers.
- Calculation of node in-degrees and out-degrees.
- Identification of maximal non-branching paths.
- Detection of isolated cycles consisting of 1-in-1-out nodes.
- Reconstruction of nucleotide sequences from graph paths.
- Preservation of repeated edges.
- Basic input validation and automated tests.

The implementation was developed for educational purposes and demonstrates fundamental graph-based genome assembly concepts.

### Learning Context

This implementation was developed as part of the *Genome Sequencing* course in the Bioinformatics Specialization offered by UC San Diego on Coursera.

In [1]:
from pathlib import Path

print("Jupyter working directory:", Path.cwd())
dataset = Path("Dataset/dataset_30189_5.txt")

print("Dataset exists:", dataset.exists())

Jupyter working directory: /mnt/c/Users/ianca/Documentos/Bioinformatics/New/bioinformatics-coursera-series/Bioinformatics_II/01-graph-based-assembly
Dataset exists: True


In [2]:

from collections import defaultdict, Counter

def build_debruijn_graph(patterns):
    """
    Construct a De Bruijn graph from a collection of k-mers.

    Each k-mer defines a directed edge from its
    (k-1)-length prefix to its (k-1)-length suffix.

    Repeated edges are preserved.

    Raises
    ------
    ValueError
        If the input is empty or contains invalid k-mers.
    """
    if not patterns:
        raise ValueError("The k-mer collection cannot be empty.")

    if any(not isinstance(pattern, str) for pattern in patterns):
        raise ValueError("All k-mers must be strings.")

    k = len(patterns[0])

    if k < 2:
        raise ValueError("K-mers must have a length of at least 2.")

    if any(len(pattern) != k for pattern in patterns):
        raise ValueError("All k-mers must have the same length.")

    graph = defaultdict(list)

    for pattern in patterns:
        prefix = pattern[:-1]
        suffix = pattern[1:]
        graph[prefix].append(suffix)

    return graph


def is_1_in_1_out(node, graph, indegree, outdegree):
    """
    Check whether a node has exactly one incoming
    and one outgoing edge.
    """
    return indegree[node] == 1 and outdegree[node] == 1


def calculate_degrees(graph):
    """
    Calculate the in-degree and out-degree of graph nodes.
    """
    indegree = defaultdict(int)
    outdegree = defaultdict(int)

    for node in graph:
        outdegree[node] = len(graph[node])

        for neighbor in graph[node]:
            indegree[neighbor] += 1

    return indegree, outdegree


def path_to_contig(path):
    """
    Convert a path of overlapping (k-1)-mers
    into a nucleotide sequence.
    """
    return path[0] + "".join(node[-1] for node in path[1:])


def find_contigs(graph):
    """
    Generate contigs from maximal non-branching paths
    and isolated cycles in a De Bruijn graph.

    Returns
    -------
    list[str]
        Contig sequences represented by the graph.
    """
    contigs = []
    indegree, outdegree = calculate_degrees(graph)

    # Track edges included in the generated paths
    visited_edges = set()

    # Step 1: Find maximal non-branching paths
    for node in graph:
        if not is_1_in_1_out(node, graph, indegree, outdegree) and outdegree[node] > 0:

            for edge_index, neighbor in enumerate(graph[node]):
                path = [node, neighbor]
                visited_edges.add((node, edge_index))

                while is_1_in_1_out(neighbor, graph, indegree, outdegree):
                    next_node = graph[neighbor][0]
                    visited_edges.add((neighbor, 0))
                    path.append(next_node)
                    neighbor = next_node

                contigs.append(path_to_contig(path))

    # Step 2: Find isolated cycles not included above
    for node in graph:
        if is_1_in_1_out(node, graph, indegree, outdegree) and (node, 0) not in visited_edges:

            cycle = [node]
            current_node = node

            while (current_node, 0) not in visited_edges:
                visited_edges.add((current_node, 0))

                next_node = graph[current_node][0]
                cycle.append(next_node)
                current_node = next_node

            contigs.append(path_to_contig(cycle))

    return contigs


# Read the input file containing space-separated k-mers
with open("Dataset/dataset_30189_5.txt", "r") as file:
    input_string = file.readline().strip()

# Convert the input string into a list of k-mers
patterns = input_string.split()

# Construct the De Bruijn graph
graph = build_debruijn_graph(patterns)

# Generate contigs
contigs = find_contigs(graph)

# Display the generated contigs
for contig in contigs:
    print(contig)

...



## Validation and Testing

The implementation was evaluated using seven test cases covering graph traversal, contig reconstruction, and input validation.

The tests verify:

1. **Branching paths:** Generation of contigs from a graph containing alternative paths.
2. **Isolated cycles:** Detection of cycles formed entirely by 1-in-1-out nodes.
3. **Repeated k-mers:** Preservation of repeated edges in the De Bruijn graph.
4. **Linear paths:** Reconstruction of a non-branching path into a single contig.
5. **K-mer conservation:** Verification that generated contigs preserve the original k-mer frequencies.
6. **Repeated-edge conservation:** Verification that duplicated k-mers are neither lost nor incorrectly added.
7. **Invalid inputs:** Rejection of empty collections, inconsistent k-mer lengths, unsupported data types, and k-mers shorter than two characters.

K-mer conservation is assessed using Python's `Counter`, allowing the comparison of k-mer frequencies rather than only unique sequences.

All seven tests passed.

In [3]:

# Test 1: Branching paths
patterns = ["ATG", "TGC", "GCA", "CAT", "ATC", "TCA"]

graph = build_debruijn_graph(patterns)
contigs = find_contigs(graph)

assert sorted(contigs) == ["ATCA", "ATGCA", "CAT"]

print("Test 1 passed: Branching paths reconstructed")
print("Generated contigs:", sorted(contigs))


# Test 2: Isolated cycle detection
patterns = ["ATG", "TGA", "GAT"]

graph = build_debruijn_graph(patterns)
contigs = find_contigs(graph)

assert len(contigs) == 1
assert contigs[0] in ["ATGAT", "TGATG", "GATGA"]

print("Test 2 passed: Isolated cycle detected")
print("Generated contigs:", contigs)


# Test 3: Preserve repeated k-mers
patterns = ["ATG", "ATG", "TGA"]

graph = build_debruijn_graph(patterns)
contigs = find_contigs(graph)

assert sorted(contigs) == ["ATG", "ATG", "TGA"]

print("Test 3 passed: Repeated k-mers preserved")
print("Generated contigs:", sorted(contigs))


# Test 4: Linear non-branching path
patterns = ["ATG", "TGC", "GCA"]

graph = build_debruijn_graph(patterns)
contigs = find_contigs(graph)

assert contigs == ["ATGCA"]

print("Test 4 passed: Linear path reconstructed")
print("Generated contigs:", contigs)


# Test 5: Verify k-mer conservation across generated contigs
patterns = ["ATG", "TGC", "GCA", "CAT", "ATC", "TCA"]

graph = build_debruijn_graph(patterns)
contigs = find_contigs(graph)

k = len(patterns[0])

# Extract k-mers from all generated contigs
reconstructed_kmers = [
    contig[i:i+k]
    for contig in contigs
    for i in range(len(contig) - k + 1)
]

assert Counter(reconstructed_kmers) == Counter(patterns)
print("Test 5 passed: All original k-mers preserved")


# Test 6: Verify k-mer conservation with repeated edges
patterns = ["ATG", "ATG", "TGA"]

graph = build_debruijn_graph(patterns)
contigs = find_contigs(graph)

k = len(patterns[0])

reconstructed_kmers = [contig[i:i+k] for contig in contigs for i in range(len(contig) - k + 1)]

assert Counter(reconstructed_kmers) == Counter(patterns)
print("Test 6 passed: Repeated k-mer frequencies preserved")


# Test 7: Reject invalid k-mer collections
invalid_inputs = [[], ["ATG", "TG"], ["A"], ["ATG", 123]]

for patterns in invalid_inputs:
    try:
        build_debruijn_graph(patterns)
    except ValueError:
        continue
    raise AssertionError(f"Invalid input was accepted: {patterns}")

print("Test 7 passed: Invalid k-mer collections rejected")

Test 1 passed: Branching paths reconstructed
Generated contigs: ['ATCA', 'ATGCA', 'CAT']
Test 2 passed: Isolated cycle detected
Generated contigs: ['ATGAT']
Test 3 passed: Repeated k-mers preserved
Generated contigs: ['ATG', 'ATG', 'TGA']
Test 4 passed: Linear path reconstructed
Generated contigs: ['ATGCA']
Test 5 passed: All original k-mers preserved
Test 6 passed: Repeated k-mer frequencies preserved
Test 7 passed: Invalid k-mer collections rejected
